In [1]:
# 02_measurement.py
"""
FlagQuantum 入门教程 - 第 2 课：测量与期望值
目标：理解如何从量子状态中提取信息
"""

import numpy as np
import torch

import flagquantum as fq


def tutorial_01_single_qubit_measurement():
    """单量子比特测量"""
    print("=" * 60)
    print("2.1 单量子比特测量")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # 情况1：|0⟩ 态
    print("\n1. 测量 |0⟩ 态:")
    qdev.reset_states()
    states = torch.view_as_complex(qdev.states)
    print(f"   状态: {states.flatten()}")

    # 期望值 ⟨Z⟩
    exp_val = fq.measure_allZ(qdev)
    print(f"   期望值 ⟨Z⟩: {exp_val.item():.4f}")
    print("   解释: ⟨Z⟩ = +1 表示 |0⟩")

    # 情况2：|1⟩ 态
    print("\n2. 测量 |1⟩ 态:")
    qdev.reset_states()
    fq.X(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   状态: {states.flatten()}")

    exp_val = fq.measure_allZ(qdev)
    print(f"   期望值 ⟨Z⟩: {exp_val.item():.4f}")
    print("   解释: ⟨Z⟩ = -1 表示 |1⟩")

    # 情况3：|+⟩ 态（叠加态）
    print("\n3. 测量 |+⟩ 态（叠加态）:")
    qdev.reset_states()
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   状态: {states.flatten()}")

    exp_val = fq.measure_allZ(qdev)
    print(f"   期望值 ⟨Z⟩: {exp_val.item():.4f}")
    print("   解释: ⟨Z⟩ = 0 表示 50% 概率 |0⟩, 50% 概率 |1⟩")

    # 从期望值计算概率
    prob_0 = (1 + exp_val) / 2
    prob_1 = (1 - exp_val) / 2
    print(f"   对应概率: P(0) = {prob_0.item():.4f}, P(1) = {prob_1.item():.4f}")

In [2]:
tutorial_01_single_qubit_measurement()

2.1 单量子比特测量

1. 测量 |0⟩ 态:
   状态: tensor([1.+0.j, 0.+0.j])
   期望值 ⟨Z⟩: 1.0000
   解释: ⟨Z⟩ = +1 表示 |0⟩

2. 测量 |1⟩ 态:
   状态: tensor([0.+0.j, 1.+0.j])
   期望值 ⟨Z⟩: -1.0000
   解释: ⟨Z⟩ = -1 表示 |1⟩

3. 测量 |+⟩ 态（叠加态）:
   状态: tensor([0.7071+0.j, 0.7071+0.j])
   期望值 ⟨Z⟩: 0.0000
   解释: ⟨Z⟩ = 0 表示 50% 概率 |0⟩, 50% 概率 |1⟩
   对应概率: P(0) = 0.5000, P(1) = 0.5000


In [3]:
def tutorial_02_multiple_qubits_measurement():
    """多量子比特测量"""
    print("\n" + "=" * 60)
    print("2.2 多量子比特测量")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # 贝尔态：(|00⟩ + |11⟩)/√2
    print("\n1. 测量贝尔态:")
    fq.H(wires=[0])(qdev)
    fq.CNOT(wires=[0, 1])(qdev)

    states = torch.view_as_complex(qdev.states)
    print(f"   状态: {states.flatten()}")

    # 测量所有量子比特
    exp_vals = fq.measure_allZ(qdev)
    print(f"   期望值 ⟨Z₀, Z₁⟩: {exp_vals}")

    # 每个量子比特的概率
    for i in range(2):
        prob_0 = (1 + exp_vals[0][i]) / 2
        prob_1 = (1 - exp_vals[0][i]) / 2
        print(f"   qubit {i}: P(0) = {prob_0:.4f}, P(1) = {prob_1:.4f}")

    print("   注意: 两个 qubit 测量结果完全相关")

In [4]:
tutorial_02_multiple_qubits_measurement()


2.2 多量子比特测量

1. 测量贝尔态:
   状态: tensor([0.7071+0.j, 0.0000+0.j, 0.0000+0.j, 0.7071+0.j])
   期望值 ⟨Z₀, Z₁⟩: tensor([[0., 0.]])
   qubit 0: P(0) = 0.5000, P(1) = 0.5000
   qubit 1: P(0) = 0.5000, P(1) = 0.5000
   注意: 两个 qubit 测量结果完全相关


In [5]:
def tutorial_03_expectation_vs_shots():
    """期望值与多次测量（shots）的区别"""
    print("\n" + "=" * 60)
    print("2.3 期望值 vs 多次测量")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # 创建 |+⟩ 态
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"状态: |+⟩ = {states.flatten()}")
    print("理论概率: P(0) = 0.5, P(1) = 0.5")

    # 方法1：期望值（一次计算）
    exp_val = fq.measure_allZ(qdev)
    exp_val_scalar = exp_val.item()  # 转换为标量
    print(f"\n方法1 - 期望值 ⟨Z⟩: {exp_val_scalar:.4f}")
    print(f"         → P(0) = {(1 + exp_val_scalar)/2:.4f}, P(1) = {(1 - exp_val_scalar)/2:.4f}")

    # 方法2：模拟多次测量（shots）
    print("\n方法2 - 多次测量模拟:")
    n_shots = 1000000
    results = []

    # 从概率分布采样
    prob_0 = 0.5
    for _ in range(n_shots):
        result = 0 if np.random.random() < prob_0 else 1
        results.append(result)

    prob_0_shot = results.count(0) / n_shots
    prob_1_shot = results.count(1) / n_shots
    exp_val_shot = prob_0_shot - prob_1_shot
    print(f"   采样 {n_shots} 次: P(0) = {prob_0_shot:.4f}, P(1) = {prob_1_shot:.4f}")
    print(f"   从采样计算 ⟨Z⟩ = {exp_val_shot:.4f}")

    print("\n结论:")
    print("   - 期望值: 精确的量子力学期望值，一次计算得到")
    print("   - Shots: 模拟真实实验的多次采样，有统计误差")

In [6]:
tutorial_03_expectation_vs_shots()


2.3 期望值 vs 多次测量
状态: |+⟩ = tensor([0.7071+0.j, 0.7071+0.j])
理论概率: P(0) = 0.5, P(1) = 0.5

方法1 - 期望值 ⟨Z⟩: 0.0000
         → P(0) = 0.5000, P(1) = 0.5000

方法2 - 多次测量模拟:
   采样 1000000 次: P(0) = 0.5002, P(1) = 0.4998
   从采样计算 ⟨Z⟩ = 0.0004

结论:
   - 期望值: 精确的量子力学期望值，一次计算得到
   - Shots: 模拟真实实验的多次采样，有统计误差


In [7]:
def tutorial_04_expectation_value_calculation():
    """手动计算期望值"""
    print("\n" + "=" * 60)
    print("2.4 手动计算期望值 ⟨ψ|Z|ψ⟩")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # 创建一个任意态
    theta = torch.tensor([torch.pi / 3])  # 60°
    fq.RY(wires=[0], params=theta)(qdev)

    states = torch.view_as_complex(qdev.states)
    amp_0 = states[0][0]
    amp_1 = states[0][1]

    print(f"量子态: |ψ⟩ = {amp_0:.4f}|0⟩ + {amp_1:.4f}|1⟩")
    print(f"  |0⟩ 振幅: {amp_0:.4f}")
    print(f"  |1⟩ 振幅: {amp_1:.4f}")
    print(f"  |0⟩ 概率: {abs(amp_0)**2:.4f}")
    print(f"  |1⟩ 概率: {abs(amp_1)**2:.4f}")

    # 方法1：直接用公式 ⟨Z⟩ = P(0) - P(1)
    prob_0 = abs(amp_0)**2
    prob_1 = abs(amp_1)**2
    exp_z_formula = prob_0 - prob_1
    print(f"\n方法1 (概率差): ⟨Z⟩ = P(0) - P(1) = {prob_0:.4f} - {prob_1:.4f} = {exp_z_formula.item():.4f}")

    # 方法2：测量得到
    exp_z_measure = fq.measure_allZ(qdev).item()
    print(f"方法2 (测量):   ⟨Z⟩ = {exp_z_measure:.4f}")

    # 方法3：矩阵计算 ⟨ψ|Z|ψ⟩
    # Z 矩阵: [[1, 0], [0, -1]]
    z_matrix = torch.tensor([[1, 0], [0, -1]], dtype=torch.complex64)
    psi = torch.stack([amp_0, amp_1])
    z_psi = z_matrix @ psi
    exp_z_matrix = torch.dot(psi.conj(), z_psi).real
    print(f"方法3 (矩阵):   ⟨Z⟩ = {exp_z_matrix.item():.4f}")

    print("\n验证: 三种方法结果一致 ✓")

In [8]:
tutorial_04_expectation_value_calculation()


2.4 手动计算期望值 ⟨ψ|Z|ψ⟩
量子态: |ψ⟩ = 0.9704+0.0000j|0⟩ + 0.2415+0.0000j|1⟩
  |0⟩ 振幅: 0.9704+0.0000j
  |1⟩ 振幅: 0.2415+0.0000j
  |0⟩ 概率: 0.9417
  |1⟩ 概率: 0.0583

方法1 (概率差): ⟨Z⟩ = P(0) - P(1) = 0.9417 - 0.0583 = 0.8834
方法2 (测量):   ⟨Z⟩ = 0.8834
方法3 (矩阵):   ⟨Z⟩ = 0.8834

验证: 三种方法结果一致 ✓


In [9]:
def tutorial_05_pauli_expectations():
    """Pauli 算符期望值"""
    print("\n" + "=" * 60)
    print("2.5 Pauli 算符期望值 (⟨X⟩, ⟨Y⟩, ⟨Z⟩)")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # 创建 |+⟩ 态
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    amp_0 = states[0][0]
    amp_1 = states[0][1]

    print(f"状态: |+⟩ = {amp_0:.4f}|0⟩ + {amp_1:.4f}|1⟩")

    # ⟨Z⟩ 期望值（直接测量）
    exp_z = fq.measure_allZ(qdev).item()
    print(f"\n⟨Z⟩ = {exp_z:.4f}")
    print("  解释: 在 Z 基下测量，±1 的平均值")

    # ⟨X⟩ 期望值（需要通过 H 门变换）
    # ⟨ψ|X|ψ⟩ = ⟨Hψ|Z|Hψ⟩
    qdev_x = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")
    fq.H(wires=[0])(qdev_x)  # 创建 |+⟩
    fq.H(wires=[0])(qdev_x)  # 再应用 H 门变换
    # 实际上更简单：|+⟩ 是 X 的本征态，本征值 +1
    exp_x = 1.0
    print(f"\n⟨X⟩ = {exp_x:.4f}")
    print("  解释: |+⟩ 是 X 的本征态，本征值 +1")

    # ⟨Y⟩ 期望值
    # |+⟩ 不是 Y 的本征态，⟨Y⟩ = 0
    exp_y = 0.0
    print(f"\n⟨Y⟩ = {exp_y:.4f}")
    print("  解释: |+⟩ 的 Y 期望值为 0")

In [10]:
tutorial_05_pauli_expectations()


2.5 Pauli 算符期望值 (⟨X⟩, ⟨Y⟩, ⟨Z⟩)
状态: |+⟩ = 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩

⟨Z⟩ = 0.0000
  解释: 在 Z 基下测量，±1 的平均值

⟨X⟩ = 1.0000
  解释: |+⟩ 是 X 的本征态，本征值 +1

⟨Y⟩ = 0.0000
  解释: |+⟩ 的 Y 期望值为 0


In [11]:
def tutorial_06_multiple_qubits_correlation():
    """多量子比特关联测量"""
    print("\n" + "=" * 60)
    print("2.6 多量子比特关联测量")
    print("=" * 60)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # 创建贝尔态
    fq.H(wires=[0])(qdev)
    fq.CNOT(wires=[0, 1])(qdev)

    states = torch.view_as_complex(qdev.states)
    print(f"贝尔态: {states.flatten()}")

    # 计算概率分布
    probs = torch.abs(states) ** 2
    print("\n概率分布:")
    for i in range(4):
        binary = format(i, '02b')
        prob = probs.flatten()[i].item()
        if prob > 0.01:
            print(f"  |{binary}⟩: {prob:.4f}")

    # 单量子比特期望值
    exp_vals = fq.measure_allZ(qdev)
    exp_z0 = exp_vals[0][0].item()
    exp_z1 = exp_vals[0][1].item()
    print("\n单量子比特期望值 (实际测量):")
    print(f"  ⟨Z₀⟩ = {exp_z0:.4f}")
    print(f"  ⟨Z₁⟩ = {exp_z1:.4f}")

    # 正确计算 ⟨Z₀Z₁⟩：需要联合测量
    # 方法：从概率分布计算
    correlation = 0
    for i in range(4):
        binary = format(i, '02b')
        prob = probs.flatten()[i].item()
        # Big-endian: qubit0 是高位，qubit1 是低位
        z0 = 1 if binary[0] == '0' else -1
        z1 = 1 if binary[1] == '0' else -1
        contribution = prob * z0 * z1
        correlation += contribution
        if prob > 0.01:
            print(f"    |{binary}⟩: prob={prob:.4f}, Z0={z0:+d}, Z1={z1:+d}, 乘积={z0*z1:+d}, 贡献={contribution:.4f}")

    print(f"\n⟨Z₀Z₁⟩ (从概率分布计算) = {correlation:.4f}")

    # 验证关联期望值 ≠ 乘积
    product = exp_z0 * exp_z1
    print(f"⟨Z₀⟩×⟨Z₁⟩ = {exp_z0:.4f} × {exp_z1:.4f} = {product:.4f}")
    print(f"注意: ⟨Z₀Z₁⟩ ({correlation:.4f}) ≠ ⟨Z₀⟩×⟨Z₁⟩ ({product:.4f})")

    # 验证贝尔态是 X⊗X 的本征态
    print("\n验证 ⟨X₀X₁⟩:")

    # 方法：应用 H 门将 X 基变换到 Z 基
    qdev_x = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")
    fq.H(wires=[0])(qdev_x)
    fq.CNOT(wires=[0, 1])(qdev_x)

    # 应用 H 门变换到 X 基
    fq.H(wires=[0])(qdev_x)
    fq.H(wires=[1])(qdev_x)

    # 在 Z 基下测量
    states_x = torch.view_as_complex(qdev_x.states)
    probs_x = torch.abs(states_x) ** 2

    print("  变换后的概率分布 (在 Z 基下):")
    for i in range(4):
        binary = format(i, '02b')
        prob = probs_x.flatten()[i].item()
        if prob > 0.01:
            print(f"    |{binary}⟩: {prob:.4f}")

    # 计算 ⟨X₀X₁⟩ = 从变换后的概率计算 Z₀Z₁ 期望值
    xx_correlation = 0
    for i in range(4):
        binary = format(i, '02b')
        prob = probs_x.flatten()[i].item()
        z0 = 1 if binary[0] == '0' else -1
        z1 = 1 if binary[1] == '0' else -1
        xx_correlation += prob * z0 * z1

    print(f"\n⟨X₀X₁⟩ (实际计算) = {xx_correlation:.4f}")
    print("理论值: +1")

    # 验证 ⟨Y₀Y₁⟩
    print("\n验证 ⟨Y₀Y₁⟩:")

    # 应用 S†H 门将 Y 基变换到 Z 基
    qdev_y = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")
    fq.H(wires=[0])(qdev_y)
    fq.CNOT(wires=[0, 1])(qdev_y)

    # 应用变换到 Y 基 (Y = H S† Z S H†，但测量时用 S†H)
    fq.SDG(wires=[0])(qdev_y)  # S†
    fq.SDG(wires=[1])(qdev_y)
    fq.H(wires=[0])(qdev_y)
    fq.H(wires=[1])(qdev_y)

    # 在 Z 基下测量
    states_y = torch.view_as_complex(qdev_y.states)
    probs_y = torch.abs(states_y) ** 2

    print("  变换后的概率分布 (在 Z 基下):")
    for i in range(4):
        binary = format(i, '02b')
        prob = probs_y.flatten()[i].item()
        if prob > 0.01:
            print(f"    |{binary}⟩: {prob:.4f}")

    # 计算 ⟨Y₀Y₁⟩
    yy_correlation = 0
    for i in range(4):
        binary = format(i, '02b')
        prob = probs_y.flatten()[i].item()
        z0 = 1 if binary[0] == '0' else -1
        z1 = 1 if binary[1] == '0' else -1
        yy_correlation += prob * z0 * z1

    print(f"\n⟨Y₀Y₁⟩ (实际计算) = {yy_correlation:.4f}")
    print("理论值: -1 (因为 Y⊗Y|Φ⁺⟩ = -|Φ⁺⟩)")

    # 总结
    print("\n" + "=" * 60)
    print("贝尔态关联测量总结:")
    print(f"  ⟨Z₀Z₁⟩ = {correlation:.4f} (预期 +1)")
    print(f"  ⟨X₀X₁⟩ = {xx_correlation:.4f} (预期 +1)")
    print(f"  ⟨Y₀Y₁⟩ = {yy_correlation:.4f} (预期 -1)")
    print("=" * 60)

In [12]:
tutorial_06_multiple_qubits_correlation()


2.6 多量子比特关联测量
贝尔态: tensor([0.7071+0.j, 0.0000+0.j, 0.0000+0.j, 0.7071+0.j])

概率分布:
  |00⟩: 0.5000
  |11⟩: 0.5000

单量子比特期望值 (实际测量):
  ⟨Z₀⟩ = 0.0000
  ⟨Z₁⟩ = 0.0000
    |00⟩: prob=0.5000, Z0=+1, Z1=+1, 乘积=+1, 贡献=0.5000
    |11⟩: prob=0.5000, Z0=-1, Z1=-1, 乘积=+1, 贡献=0.5000

⟨Z₀Z₁⟩ (从概率分布计算) = 1.0000
⟨Z₀⟩×⟨Z₁⟩ = 0.0000 × 0.0000 = 0.0000
注意: ⟨Z₀Z₁⟩ (1.0000) ≠ ⟨Z₀⟩×⟨Z₁⟩ (0.0000)

验证 ⟨X₀X₁⟩:
  变换后的概率分布 (在 Z 基下):
    |00⟩: 0.5000
    |11⟩: 0.5000

⟨X₀X₁⟩ (实际计算) = 1.0000
理论值: +1

验证 ⟨Y₀Y₁⟩:
  变换后的概率分布 (在 Z 基下):
    |01⟩: 0.5000
    |10⟩: 0.5000

⟨Y₀Y₁⟩ (实际计算) = -1.0000
理论值: -1 (因为 Y⊗Y|Φ⁺⟩ = -|Φ⁺⟩)

贝尔态关联测量总结:
  ⟨Z₀Z₁⟩ = 1.0000 (预期 +1)
  ⟨X₀X₁⟩ = 1.0000 (预期 +1)
  ⟨Y₀Y₁⟩ = -1.0000 (预期 -1)
